# kinilum · 02 · Fuentes de datos y EDA

**NASA Space Apps 2026 · reto Field Shift** · v0.1 · 14 de noviembre de 2026 · Jorge Vargas

Este notebook hace tres cosas, en este orden:

1. **Define el contrato de datos** que debe cumplir todo valor que entre a kinilum (fuente, resolución nativa, nivel de soporte, método, calidad, procedencia).
2. **Conecta las fuentes** por API, una función por fuente, con caché y sin secretos en el código.
3. **Ejecuta el EDA en nueve etapas (E0 a E8)** y escribe un informe que una persona o un agente puede leer: `eda_report.json`, `eda_report.md` y un borrador del paquete de datos.

**Regla que no se rompe: nada simulado.** Una fuente que no llega se muestra como `por integrar` o `sin dato`. El modo `fixture` existe solo para probar que el código corre; todo lo que produce lleva la marca `PRUEBA` y se escribe en otra carpeta.

**En Google Colab:** abra el cuaderno, guarde los tokens que tenga en *Secretos* con los nombres `EARTHDATA_TOKEN`, `CDS_API_KEY` e `INEGI_TOKEN` (con acceso al cuaderno activado) y ejecute todo. Sin tokens, las fuentes que los piden quedan `por integrar`. Las fuentes abiertas funcionan sin ellos.

**Cómo se ejecuta sin abrirlo:** `papermill kinilum_eda_00.ipynb out.ipynb -p MODE live -p LON -108.45673 -p LAT 25.50826`. La celda de parámetros está marcada para eso.

**Estado de cada conector** (se verificó contra documentación pública; ninguna llamada se ejecutó en vivo al escribir esto):

| Conector | Acceso | Estado |
| --- | --- | --- |
| NASA POWER diario | abierto, sin llave | listo para probar |
| Open-Meteo archivo (reanálisis ERA5) | abierto, sin llave | listo para probar |
| SoilGrids propiedades | abierto, sin llave | listo para probar |
| HLS por CMR-STAC (solo búsqueda de escenas) | abierto para buscar; descargar pide token Earthdata | listo para probar |
| Sentinel-2 por Earth Search (respaldo) | abierto, sin llave | listo para probar |
| IMERG diario | token Earthdata + `earthaccess` | esqueleto |
| NASADEM, MODIS, SMAP, ECOSTRESS, GRACE | token Earthdata | esqueleto, se descubren con `cmr_find_collection` |
| ERA5-Land por CDS | llave de CDS | esqueleto |
| INEGI indicadores | token INEGI | esqueleto, contexto |

## 0. Parámetros

In [ ]:
MODE = "live"                  # "live" llama a las APIs. "fixture" usa datos de PRUEBA para probar el código.
PLOT_NAME = "parcela-ejemplo"
LON, LAT = -108.45673, 25.50826  # EJEMPLO sorteado cerca de Guasave, Sinaloa. Reemplazar por el centro de la parcela real.
PLOT_SIDE_M = 200              # lado de la parcela en metros (supuesto para el ejemplo)
ELEV_M = 15.0                  # SUPUESTO. Se reemplaza con NASADEM cuando ese conector esté integrado.
START, END = "2016-01-01", "2025-12-31"
CLOUD_MAX = 30                 # % de nubes aceptado en escenas HLS y Sentinel-2 (propuesta, por calibrar)
WET_MM = 1.0                   # umbral de día con lluvia en mm (propuesta)
ENABLED = ["power", "openmeteo", "soilgrids", "hls_stac", "s2_earthsearch",
           "imerg", "nasadem", "modis", "smap", "ecostress", "grace", "era5land"]
CACHE_DIR = "cache"
OUT_DIR = None                 # None: out/ en modo live, out_fixture/ en modo fixture

## 1. Preparación

In [ ]:
import os, json, hashlib, time, math, datetime as dt
from dataclasses import dataclass, field, asdict
from pathlib import Path
import numpy as np, pandas as pd, requests
import matplotlib
import matplotlib.pyplot as plt

assert MODE in ("live", "fixture"), "MODE debe ser 'live' o 'fixture'"
FIXTURE = MODE == "fixture"
OUT = Path(OUT_DIR or ("out_fixture" if FIXTURE else "out")); OUT.mkdir(parents=True, exist_ok=True)
CACHE = Path(CACHE_DIR); CACHE.mkdir(parents=True, exist_ok=True)
UA = "kinilum-eda/0.1 (NASA Space Apps 2026; contacto: equipo kinilum)"
NOW = lambda: dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "earthaccess", "xarray", "netCDF4"], check=False)
    for _k in ("EARTHDATA_TOKEN", "CDS_API_KEY", "INEGI_TOKEN"):
        try:
            _v = userdata.get(_k)
            if _v: os.environ[_k] = _v
        except Exception:
            pass


SECRETS = {"EARTHDATA_TOKEN": "NASA Earthdata (IMERG, NASADEM, MODIS, SMAP, ECOSTRESS, GRACE, descarga HLS)",
           "CDS_API_KEY": "Copernicus CDS (ERA5-Land)",
           "INEGI_TOKEN": "INEGI indicadores"}
def secret(name): return os.environ.get(name)
print({k: ("presente" if secret(k) else "ausente") for k in SECRETS})
print("MODO:", MODE, "| salida:", OUT)
if FIXTURE: print("*** DATOS DE PRUEBA: nada de lo que sigue son resultados ***")

## 2. Contrato de datos

Cada valor que llega a kinilum lleva la misma ficha. Las reglas de **nivel de soporte** salen del PRD: parcela ≤ 70 m, entorno 250 m a 1 km, zona ≈ 10 km, región ≥ 50 km. El PRD deja huecos entre esos rangos; los cortes de abajo son una **propuesta** para cerrarlos.

In [ ]:
class SourcePending(Exception):
    "La fuente aún no está integrada o falta una credencial. Se informa como 'por integrar'."
class SourceError(Exception):
    "La fuente respondió con error o no respondió. Se informa como 'sin dato'."

def support_level(native_res_m):
    "Propuesta de cortes: <=70 m parcela; <=1 km entorno; <=15 km zona; mayor, región."
    if native_res_m is None: return "sin_dato"
    if native_res_m <= 70: return "parcela"
    if native_res_m <= 1000: return "entorno"
    if native_res_m <= 15000: return "zona"
    return "region"

@dataclass
class Provenance:
    source: str; product: str; endpoint: str; params: dict
    requested_at: str = field(default_factory=NOW)
    response_sha256: str = ""; license: str = ""; citation: str = ""
    fixture: bool = False

@dataclass
class Record:
    "Un valor con su contrato. p10 y p90 pueden ser otros cuantiles si la fuente no da esos; se dice en `assumptions`."
    name: str; value: float | None; unit: str; p10: float | None; p90: float | None
    source: str; native_res_m: float; derived_res_m: float
    support_level: str; method: str          # medido | interpolado | modelado | supuesto
    quality: str; assumptions: list; provenance: dict

def make_record(name, value, unit, p10, p90, prov, native_res_m, method, quality="sin evaluar",
                assumptions=None, derived_res_m=None):
    num = lambda x: None if x is None or (isinstance(x, float) and math.isnan(x)) else float(x)   # sin dato: None, nunca float(None)
    return Record(name, num(value), unit, num(p10), num(p90), prov.source, native_res_m,
                  derived_res_m or native_res_m, support_level(native_res_m), method,
                  "PRUEBA" if prov.fixture else quality, assumptions or [], asdict(prov))

def plot_bbox(lon, lat, side_m):
    dlat = side_m / 2 / 111_320
    dlon = side_m / 2 / (111_320 * math.cos(math.radians(lat)))
    return [lon - dlon, lat - dlat, lon + dlon, lat + dlat]

def clean_pixels(side_m, res_m, buffer_m=30):
    "Píxeles completos dentro de la parcela tras descartar un borde interno (regla del PRD: búfer de 30 m)."
    inner = max(side_m - 2 * buffer_m, 0)
    return int(inner // res_m) ** 2

print("Píxeles limpios HLS (30 m) en una parcela de", PLOT_SIDE_M, "m:", clean_pixels(PLOT_SIDE_M, 30))
print("Píxeles limpios MODIS (250 m):", clean_pixels(PLOT_SIDE_M, 250))

## 3. Cliente HTTP con caché

Una sola función hace las llamadas. Guarda la respuesta y su hash en `cache/` para no repetir peticiones (POWER avisa que puede bloquear a quien pide lo mismo muchas veces) y para poder demostrar de dónde salió cada dato. **No registra encabezados**, que es donde viaja el token.

In [ ]:
def cached_json(url, params=None, method="GET", body=None, headers=None, tries=3, timeout=90, sensitive=False):
    "sensitive=True: la URL lleva un secreto (p. ej. token en la ruta). No se guarda en caché ni se imprime."
    from urllib.parse import urlparse
    shown = urlparse(url).netloc if sensitive else url
    key = hashlib.sha256(json.dumps([method, url, params, body], sort_keys=True, default=str).encode()).hexdigest()
    f = CACHE / f"{key}.json"
    if f.exists() and not sensitive:
        c = json.loads(f.read_text()); return c["body"], c["sha256"], c["url"]
    last = None
    for i in range(tries):
        try:
            r = requests.request(method, url, params=params, json=body, timeout=timeout,
                                 headers={"User-Agent": UA, **(headers or {})})
            if r.status_code == 429 or r.status_code >= 500:
                raise SourceError(f"HTTP {r.status_code}")
            r.raise_for_status()
            sha = hashlib.sha256(r.content).hexdigest()
            data = r.json()
            if not sensitive: f.write_text(json.dumps({"body": data, "sha256": sha, "url": r.url, "at": NOW()}))
            return data, sha, r.url
        except (requests.RequestException, SourceError, ValueError) as e:
            last = e; time.sleep(2 ** i)
    raise SourceError(f"{shown}: {type(last).__name__}" + ("" if sensitive else f": {str(last)[:160]}"))

@dataclass
class Fetch:
    df: pd.DataFrame; prov: Provenance; units: dict = field(default_factory=dict); notes: list = field(default_factory=list)

## 4. Conectores

Todos devuelven `Fetch` y lanzan `SourcePending` o `SourceError`; nunca devuelven datos inventados. Los conectores marcados *esqueleto* dicen exactamente qué falta. **Para un agente:** cada función es una unidad de trabajo; para añadir una fuente se copia una y se cumple la lista de la sección 6 del documento 2.

### 4.1 NASA POWER (diario, punto)
`GET /api/temporal/daily/point` con `community=AG`. Valor de relleno `-999`. Resolución nativa 0.5° × 0.625°, que a esta latitud son unos 55 por 65 km: nivel de soporte **región**.

In [ ]:
POWER_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"
POWER_PARS = ["PRECTOTCORR", "T2M", "T2M_MAX", "T2M_MIN", "RH2M", "WS2M", "ALLSKY_SFC_SW_DWN"]

def fetch_power(lon, lat, start, end, pars=POWER_PARS):
    if FIXTURE: return fx_power(start, end, pars)
    frames, units, sha_all = [], {}, []
    for y in range(int(start[:4]), int(end[:4]) + 1):          # una petición por año
        s, e = max(start, f"{y}-01-01"), min(end, f"{y}-12-31")
        body, sha, url = cached_json(POWER_URL, params={
            "parameters": ",".join(pars), "community": "AG", "longitude": lon, "latitude": lat,
            "start": s.replace("-", ""), "end": e.replace("-", ""), "format": "JSON"})
        par = body["properties"]["parameter"]
        d = pd.DataFrame(par); d.index = pd.to_datetime(d.index, format="%Y%m%d"); frames.append(d)
        units.update({k: v.get("units") for k, v in body.get("parameters", {}).items()}); sha_all.append(sha)
    df = pd.concat(frames).sort_index().replace(-999, np.nan)
    prov = Provenance("NASA POWER", "daily point (AG)", POWER_URL, {"lon": lon, "lat": lat, "start": start, "end": end, "pars": pars},
                      response_sha256=hashlib.sha256("".join(sha_all).encode()).hexdigest(),
                      license="Datos abiertos NASA", citation="NASA POWER Project, power.larc.nasa.gov")
    return Fetch(df, prov, units, ["Resolución nativa ~0.5° x 0.625° (región)"])

### 4.2 Open-Meteo archivo (externa a NASA)
Reanálisis ERA5 servido sin llave. Sirve de **contraste independiente** para lluvia y de referencia para ET0 (`et0_fao_evapotranspiration`). Se pide en UTC para que los días coincidan con POWER. Al ser un servicio de terceros, el contrato registra el modelo que se usó; por defecto el servicio elige el mejor modelo disponible, así que conviene fijar `models` cuando se compare en serio (por verificar con la documentación del servicio).

In [ ]:
OM_URL = "https://archive-api.open-meteo.com/v1/archive"
OM_DAILY = ["precipitation_sum", "et0_fao_evapotranspiration", "temperature_2m_max", "temperature_2m_min", "shortwave_radiation_sum"]

def fetch_openmeteo(lon, lat, start, end, daily=OM_DAILY, models=None):
    if FIXTURE: return fx_openmeteo(start, end)
    p = {"latitude": lat, "longitude": lon, "start_date": start, "end_date": end, "daily": ",".join(daily), "timezone": "UTC"}
    if models: p["models"] = models
    body, sha, url = cached_json(OM_URL, params=p)
    d = pd.DataFrame(body["daily"]).set_index("time"); d.index = pd.to_datetime(d.index)
    prov = Provenance("Open-Meteo (ERA5)", "archive daily", OM_URL, p, response_sha256=sha,
                      license="CC BY 4.0 (Open-Meteo)", citation="Open-Meteo.com; datos ERA5 de Copernicus/ECMWF")
    return Fetch(d, prov, body.get("daily_units", {}), ["Resolución nativa depende del modelo; registrar `models`."])

### 4.3 SoilGrids (externa a NASA)
Propiedades del suelo a 250 m. La API entrega enteros escalados: se divide entre `d_factor`. SoilGrids da cuantiles 5 y 95, **no** 10 y 90: se usan como límites del rango y se anota en los supuestos.

In [ ]:
SG_URL = "https://rest.isric.org/soilgrids/v2.0/properties/query"
SG_PROPS = ["clay", "sand", "silt", "soc", "phh2o", "bdod"]
SG_DEPTHS = ["0-5cm", "5-15cm", "15-30cm", "30-60cm", "60-100cm"]
SG_RADII_M = (250, 500, 1000)   # si el punto no trae valores, se prueban celdas a estas distancias (propuesta)
SG_PAUSE_S = 12                 # SoilGrids limita a ~5 peticiones por minuto
_DIRS = ["norte", "noreste", "este", "sureste", "sur", "suroeste", "oeste", "noroeste"]

def _sg_query(lon, lat, props, depths):
    p = {"lon": lon, "lat": lat, "property": props, "depth": depths, "value": ["mean", "Q0.05", "Q0.95"]}
    body, sha, _ = cached_json(SG_URL, params=p)
    return p, body, sha

def _sg_has_values(body):
    "SoilGrids responde 200 con todo en null donde no hay celda mapeada (p. ej. sobre roca caliza): eso es «sin dato», no «ok»."
    return any(d["values"].get("mean") is not None for l in body["properties"]["layers"] for d in l["depths"])

def _sg_candidates(lon, lat):
    for r in SG_RADII_M:
        for k, name in enumerate(_DIRS):
            a = math.radians(45 * k)
            yield r, name, lon + r * math.sin(a) / (111_320 * math.cos(math.radians(lat))), lat + r * math.cos(a) / 111_320

def fetch_soilgrids(lon, lat, props=SG_PROPS, depths=SG_DEPTHS):
    if FIXTURE: return fx_soil()
    p, body, sha = _sg_query(lon, lat, props, depths)
    notes = []
    if not _sg_has_values(body):
        found = None
        for r, name, lo, la in _sg_candidates(lon, lat):
            time.sleep(SG_PAUSE_S)
            _, probe, _ = _sg_query(lo, la, ["clay"], ["0-5cm"])
            if _sg_has_values(probe):
                p, body, sha = _sg_query(lo, la, props, depths); found = (r, name, lo, la); break
        if not found:
            raise SourceError(f"SoilGrids no tiene valores en el punto ni a {SG_RADII_M[-1]} m a la redonda: sin dato")
        r, name, lo, la = found
        p = {**p, "fallback": {"distance_m": r, "direction": name, "lon": round(lo, 5), "lat": round(la, 5)}}
        notes.append(f"El punto no tiene valores: se usó la celda válida más cercana, a {r} m al {name}.")
    rows = []
    for layer in body["properties"]["layers"]:
        um = layer.get("unit_measure", {}); fac = um.get("d_factor")
        if not fac: notes.append(f"{layer['name']}: sin d_factor, valores sin convertir"); fac = 1
        for d in layer["depths"]:
            v = d["values"]
            rows.append({"property": layer["name"], "depth": d["label"], "units": um.get("target_units"),
                         **{k: (None if x is None else x / fac) for k, x in v.items()}})
    prov = Provenance("SoilGrids (ISRIC)", "properties/query v2.0", SG_URL, p, response_sha256=sha,
                      license="CC BY 4.0 (ISRIC)", citation="Poggio et al. 2021, SoilGrids 2.0, SOIL 7")
    return Fetch(pd.DataFrame(rows), prov, notes=notes + ["Rango = cuantiles 5 y 95, no 10 y 90."])

### 4.4 HLS por CMR-STAC y Sentinel-2 por Earth Search (solo escenas)
Aquí **no se descargan píxeles**: se buscan escenas y se lee su `eo:cloud_cover`. Eso basta para saber cuántas observaciones útiles hay por mes. Los identificadores de colección de CMR-STAC cambiaron de formato (`HLSL30.v2.0` pasó a `HLSL30_2.0`), así que el código los **descubre** en lugar de fijarlos. Earth Search sirve Sentinel-2 sin credencial y sirve de respaldo si Earthdata no está disponible; no reemplaza a HLS en el contrato (es otra fuente).

In [ ]:
CMR_STAC = "https://cmr.earthdata.nasa.gov/stac/LPCLOUD"
EARTH_SEARCH = "https://earth-search.aws.element84.com/v1"

def stac_search(base, collections, bbox, start, end, limit=200, max_pages=60):
    body = {"collections": collections, "bbox": bbox, "datetime": f"{start}T00:00:00Z/{end}T23:59:59Z", "limit": limit}
    rows, pages, sha_all = [], 0, []
    url, method, payload = f"{base}/search", "POST", body
    while url and pages < max_pages:
        data, sha, _ = cached_json(url, method=method, body=payload); sha_all.append(sha); pages += 1
        for f in data.get("features", []):
            pr = f.get("properties", {})
            rows.append({"id": f.get("id"), "collection": f.get("collection"), "datetime": pr.get("datetime"),
                         "cloud": pr.get("eo:cloud_cover")})
        nxt = next((l for l in data.get("links", []) if l.get("rel") == "next"), None)
        if nxt: url, method, payload = nxt["href"], nxt.get("method", "GET"), nxt.get("body")
        else: url = None
    df = pd.DataFrame(rows)
    if len(df): df["datetime"] = pd.to_datetime(df["datetime"], utc=True, format="ISO8601")
    if len(df): df = df.drop_duplicates("id").reset_index(drop=True)
    return df, hashlib.sha256("".join(sha_all).encode()).hexdigest(), pages >= max_pages

def hls_collection_ids():
    data, _, _ = cached_json(f"{CMR_STAC}/collections", params={"limit": 200})
    ids = [c["id"] for c in data.get("collections", []) if c["id"].startswith(("HLSL30", "HLSS30")) and "_VI_" not in c["id"]]   # _VI_ repite las mismas escenas
    # preferir la versión 2.0 si hay varias
    return sorted(i for i in ids if "2.0" in i) or ids

def fetch_hls_stac(lon, lat, start, end, side_m=PLOT_SIDE_M):
    if FIXTURE: return fx_stac("HLS (CMR-STAC)", start, end, every=4)
    ids = hls_collection_ids()
    if not ids: raise SourceError("CMR-STAC no devolvió colecciones HLS")
    df, sha, cut = stac_search(CMR_STAC, ids, plot_bbox(lon, lat, side_m), start, end, limit=50)   # con páginas más grandes CMR-STAC responde 502
    prov = Provenance("HLS (NASA LP DAAC)", "+".join(ids), f"{CMR_STAC}/search", {"bbox": plot_bbox(lon, lat, side_m)}, response_sha256=sha,
                      license="Datos abiertos NASA", citation="NASA LP DAAC, HLS v2.0")
    return Fetch(df, prov, notes=["Solo metadatos de escenas; sin píxeles."] + (["Paginación cortada en max_pages"] if cut else []))

def fetch_s2_earthsearch(lon, lat, start, end, side_m=PLOT_SIDE_M):
    if FIXTURE: return fx_stac("Sentinel-2 (Earth Search)", start, end, every=5)
    df, sha, cut = stac_search(EARTH_SEARCH, ["sentinel-2-l2a"], plot_bbox(lon, lat, side_m), start, end)
    prov = Provenance("Sentinel-2 L2A (Earth Search)", "sentinel-2-l2a", f"{EARTH_SEARCH}/search", {"bbox": plot_bbox(lon, lat, side_m)},
                      response_sha256=sha, license="Copernicus Sentinel data (libre)", citation="Contains modified Copernicus Sentinel data; Element 84 Earth Search")
    return Fetch(df, prov, notes=["Solo metadatos de escenas; sin píxeles."] + (["Paginación cortada en max_pages"] if cut else []))

### 4.5 Descubrir productos de NASA en CMR (sin credencial)
Para no adivinar nombres cortos ni versiones, esta función los **busca**. Un agente que deba integrar IMERG, NASADEM, MODIS, SMAP, ECOSTRESS o GRACE empieza aquí.

In [ ]:
CMR = "https://cmr.earthdata.nasa.gov/search"
def cmr_find_collection(keyword, n=8):
    data, _, _ = cached_json(f"{CMR}/collections.json", params={"keyword": keyword, "page_size": n})
    return pd.DataFrame([{"short_name": e.get("short_name"), "version": e.get("version_id"), "title": e.get("title"),
                          "provider": e.get("data_center"), "start": e.get("time_start")} for e in data["feed"]["entry"]])

# Ejemplo (requiere red): cmr_find_collection("GPM IMERG Final daily")

### 4.6 Conectores con credencial (esqueletos)
Cada uno declara **qué producto, qué variable y qué falta**. Hasta que se implementen lanzan `SourcePending`. El caso de IMERG tiene el flujo completo con `earthaccess`; por verificar el nombre de la variable en la versión 07.

In [ ]:
def _need_token(name="EARTHDATA_TOKEN"):
    if not secret(name): raise SourcePending(f"falta {name}")

def fetch_imerg(lon, lat, start, end):
    "GPM IMERG Final diario 0.1° (GPM_3IMERGDF v07). Latencia de la corrida Final: hasta ~4 meses."
    if FIXTURE: return fx_imerg(start, end)
    _need_token()
    try: import earthaccess, xarray as xr
    except ImportError: raise SourcePending("pip install earthaccess xarray netcdf4")
    earthaccess.login(strategy="environment")
    res = earthaccess.search_data(short_name="GPM_3IMERGDF", version="07", temporal=(start, end),
                                  bounding_box=(lon - .05, lat - .05, lon + .05, lat + .05))
    if not res: raise SourceError("IMERG: búsqueda sin granulos")
    files = earthaccess.open(res); rows = []
    for f in files:
        ds = xr.open_dataset(f); var = next((v for v in ("precipitation", "precipitationCal") if v in ds), None)
        if var is None: raise SourceError(f"IMERG: variable de lluvia no encontrada en {list(ds.data_vars)}")
        pt = ds[var].sel(lon=lon, lat=lat, method="nearest")
        rows.append({"date": pd.to_datetime(ds.time.values[0]).normalize(), "precip_mm_day": float(pt.values.squeeze())})
    df = pd.DataFrame(rows).set_index("date").sort_index()
    prov = Provenance("GPM IMERG Final (NASA GES DISC)", "GPM_3IMERGDF v07", "earthaccess", {"lon": lon, "lat": lat, "start": start, "end": end},
                      license="Datos abiertos NASA", citation="Huffman et al., GPM IMERG Final Daily V07, GES DISC")
    return Fetch(df, prov, {"precip_mm_day": "mm/day"}, ["0.1° (zona). Ver conteo de muestras válidas por día en el archivo original."])

def fetch_nasadem(lon, lat, **k):
    if FIXTURE: raise SourcePending("fixture: sin elevación de prueba")
    _need_token(); raise SourcePending("esqueleto: NASADEM_HGT (buscar versión con cmr_find_collection('NASADEM')); leer elevación y pendiente de la celda de 30 m")
def fetch_modis(lon, lat, **k):
    if FIXTURE: return fx_ndvi()
    _need_token(); raise SourcePending("esqueleto: MOD13Q1 (NDVI 250 m, 16 días); extraer serie del píxel de la parcela y su calidad (campo de confiabilidad)")
def fetch_smap(lon, lat, **k):
    _need_token(); raise SourcePending("esqueleto: SMAP L4 (9 km, 3 h); buscar producto con cmr_find_collection('SMAP L4 soil moisture')")
def fetch_ecostress(lon, lat, **k):
    _need_token(); raise SourcePending("esqueleto: ECOSTRESS (70 m, pasadas irregulares); cubrir con POWER si faltan pasadas")
def fetch_grace(lon, lat, **k):
    _need_token(); raise SourcePending("esqueleto: GRACE/GRACE-FO mascones (~300 km, mensual); P1, se corta primero")
def fetch_era5land(lon, lat, start, end):
    if not secret("CDS_API_KEY"): raise SourcePending("falta CDS_API_KEY (la cola de CDS puede tardar)")
    raise SourcePending("esqueleto: cdsapi con el conjunto 'reanalysis-era5-land-timeseries'; verificar claves de la petición en la documentación de CDS")
def fetch_inegi(indicator, geo="00", token_env="INEGI_TOKEN"):
    "Contexto socioeconómico, no agronómico. Sintaxis tomada de la guía del INEGI."
    _need_token(token_env)
    u = f"https://www.inegi.org.mx/app/api/indicadores/desarrolladores/jsonxml/INDICATOR/{indicator}/es/{geo}/false/BIE/2.0/{secret(token_env)}"
    return cached_json(u, params={"type": "json"}, sensitive=True)   # el token va en la ruta: sin caché y sin URL en errores

## 5. Datos de PRUEBA (solo para probar el código)

Este bloque **solo se usa con `MODE = "fixture"`**. Genera series con forma plausible para que las nueve etapas corran de punta a punta. No representa ningún lugar real. Todo lo que salga de aquí queda marcado `PRUEBA` en gráficas, informe y paquete.

In [ ]:
def _rng(): return np.random.default_rng(7)
def _days(a, b): return pd.date_range(a, b, freq="D")
def _fx_prov(src, prod): return Provenance(src, prod, "fixture", {}, fixture=True, license="n/a", citation="PRUEBA")

def fx_power(start, end, pars):
    r, d = _rng(), _days(start, end); doy = d.dayofyear.values
    wet = (np.sin((doy - 120) / 365 * 2 * np.pi) > 0.1)
    rain = np.where(r.random(len(d)) < np.where(wet, .45, .08), r.gamma(1.2, 9, len(d)), 0.0)
    tmean = 26 + 3 * np.sin((doy - 100) / 365 * 2 * np.pi) + r.normal(0, .8, len(d))
    df = pd.DataFrame({"PRECTOTCORR": rain, "T2M": tmean, "T2M_MAX": tmean + 5 + r.normal(0, .6, len(d)),
                       "T2M_MIN": tmean - 5 + r.normal(0, .6, len(d)),
                       "RH2M": np.clip(75 + 8 * wet + r.normal(0, 5, len(d)), 30, 100), "WS2M": np.abs(r.normal(2.0, .6, len(d))),
                       "ALLSKY_SFC_SW_DWN": np.clip(19 - 4 * wet + r.normal(0, 2.5, len(d)), 3, 30)}, index=d)[pars]
    df.iloc[40:44] = np.nan   # un hueco, para probar las etapas de calidad
    return Fetch(df, _fx_prov("NASA POWER", "PRUEBA"), {"PRECTOTCORR": "mm/day", "T2M": "C", "T2M_MAX": "C", "T2M_MIN": "C",
                 "RH2M": "%", "WS2M": "m/s", "ALLSKY_SFC_SW_DWN": "MJ/m^2/day"}, ["DATOS DE PRUEBA"])

def fx_openmeteo(start, end):
    p = fx_power(start, end, POWER_PARS).df; r = np.random.default_rng(11)
    df = pd.DataFrame({"precipitation_sum": np.clip(p["PRECTOTCORR"].fillna(0) * r.normal(1.0, .35, len(p)), 0, None),
                       "et0_fao_evapotranspiration": np.clip(4.2 + .08 * (p["T2M"].fillna(26) - 26) * 5 + r.normal(0, .4, len(p)), .5, None),
                       "temperature_2m_max": p["T2M_MAX"], "temperature_2m_min": p["T2M_MIN"],
                       "shortwave_radiation_sum": p["ALLSKY_SFC_SW_DWN"]}, index=p.index)
    return Fetch(df, _fx_prov("Open-Meteo (ERA5)", "PRUEBA"), {"precipitation_sum": "mm"}, ["DATOS DE PRUEBA"])

def fx_imerg(start, end):
    p = fx_power(start, end, POWER_PARS).df; r = np.random.default_rng(13)
    df = pd.DataFrame({"precip_mm_day": np.clip(p["PRECTOTCORR"].fillna(0) * r.normal(1.0, .5, len(p)), 0, None)}, index=p.index)
    return Fetch(df, _fx_prov("GPM IMERG Final", "PRUEBA"), {"precip_mm_day": "mm/day"}, ["DATOS DE PRUEBA"])

def fx_soil():
    rows = [{"property": pr, "depth": dp, "mean": m, "Q0.05": m * .8, "Q0.95": m * 1.2, "units": u}
            for pr, m, u in (("clay", 38, "%"), ("sand", 22, "%"), ("silt", 40, "%"), ("soc", 18, "g/kg"), ("phh2o", 7.1, "pH"), ("bdod", 1.1, "g/cm3"))
            for dp in SG_DEPTHS]
    return Fetch(pd.DataFrame(rows), _fx_prov("SoilGrids (ISRIC)", "PRUEBA"), notes=["DATOS DE PRUEBA"])

def fx_stac(name, start, end, every):
    r = np.random.default_rng(17); d = pd.date_range(start, end, freq=f"{every}D", tz="UTC")
    doy = d.dayofyear.values; wet = np.sin((doy - 120) / 365 * 2 * np.pi) > 0.1
    df = pd.DataFrame({"id": [f"fx-{i}" for i in range(len(d))], "collection": "PRUEBA", "datetime": d,
                       "cloud": np.clip(np.where(wet, r.beta(3, 2, len(d)), r.beta(1, 4, len(d))) * 100, 0, 100)})
    return Fetch(df, _fx_prov(name, "PRUEBA"), notes=["DATOS DE PRUEBA"])

def fx_ndvi():
    d = pd.date_range(START, END, freq="16D"); doy = d.dayofyear.values
    v = 0.30 + 0.35 * (np.sin((doy - 140) / 365 * 2 * np.pi) > 0) * np.abs(np.sin((doy - 140) / 365 * 2 * np.pi)) + np.random.default_rng(3).normal(0, .03, len(d))
    return Fetch(pd.DataFrame({"ndvi": np.clip(v, 0, 1)}, index=d), _fx_prov("MODIS NDVI", "PRUEBA"), notes=["DATOS DE PRUEBA"])

## 6. Catálogo de fuentes

El catálogo es **un dato, no código**. Es el mismo catálogo que consume la tarea «Obtener una fuente» del diagrama BPMN. Agregar una fuente es agregar una fila y su conector.

In [ ]:
CATALOG = pd.DataFrame([
 # id, nombre, origen, rol, acceso, resolución nativa (m), cadencia, prioridad, conector
 ("power", "NASA POWER", "NASA", "núcleo", "abierto", 55000, "diaria", "P0", fetch_power),
 ("imerg", "GPM IMERG Final", "NASA", "núcleo", "token Earthdata", 11000, "diaria (latencia ~4 meses)", "P0", fetch_imerg),
 ("hls_stac", "HLS (Landsat + Sentinel-2)", "NASA", "núcleo", "buscar abierto; descargar con token", 30, "2 a 3 días", "P0", fetch_hls_stac),
 ("nasadem", "NASADEM / SRTM", "NASA", "núcleo", "token Earthdata", 30, "estático", "P0", fetch_nasadem),
 ("modis", "MODIS NDVI MOD13Q1", "NASA", "quinta fuente", "token Earthdata", 250, "16 días", "P0", fetch_modis),
 ("smap", "SMAP L4", "NASA", "recortable", "token Earthdata", 9000, "3 horas", "P0", fetch_smap),
 ("ecostress", "ECOSTRESS", "NASA", "recortable", "token Earthdata", 70, "irregular", "P0", fetch_ecostress),
 ("grace", "GRACE / GRACE-FO", "NASA", "recortable", "token Earthdata", 300000, "mensual", "P1", fetch_grace),
 ("openmeteo", "Open-Meteo archivo (ERA5)", "externa", "contraste", "abierto", 25000, "diaria", "propuesta", fetch_openmeteo),
 ("soilgrids", "SoilGrids", "externa", "suelo", "abierto", 250, "estático", "propuesta", fetch_soilgrids),
 ("s2_earthsearch", "Sentinel-2 L2A (Earth Search)", "externa", "respaldo de HLS", "abierto", 10, "5 días", "propuesta", fetch_s2_earthsearch),
 ("era5land", "ERA5-Land (CDS)", "externa", "contraste", "llave CDS", 9000, "horaria", "propuesta", fetch_era5land),
], columns=["id", "nombre", "origen", "rol", "acceso", "res_nativa_m", "cadencia", "prioridad", "conector"])
CATALOG["soporte"] = CATALOG["res_nativa_m"].map(support_level)
CATALOG.drop(columns="conector").to_json(OUT / "sources.json", orient="records", force_ascii=False, indent=1)
CATALOG.drop(columns="conector")

## 7. Carga: cada fuente termina en uno de tres estados

`ok` (llegó), `por integrar` (falta credencial o conector) o `sin dato` (error de la fuente). **Nunca** se rellena con datos inventados.

In [ ]:
DATA, STATUS = {}, {}
CALL = {"power": lambda: fetch_power(LON, LAT, START, END), "openmeteo": lambda: fetch_openmeteo(LON, LAT, START, END),
        "soilgrids": lambda: fetch_soilgrids(LON, LAT), "hls_stac": lambda: fetch_hls_stac(LON, LAT, START, END),
        "s2_earthsearch": lambda: fetch_s2_earthsearch(LON, LAT, START, END), "imerg": lambda: fetch_imerg(LON, LAT, START, END),
        "nasadem": lambda: fetch_nasadem(LON, LAT), "modis": lambda: fetch_modis(LON, LAT), "smap": lambda: fetch_smap(LON, LAT),
        "ecostress": lambda: fetch_ecostress(LON, LAT), "grace": lambda: fetch_grace(LON, LAT),
        "era5land": lambda: fetch_era5land(LON, LAT, START, END)}
for sid in [s for s in CATALOG["id"] if s in ENABLED]:
    try:
        DATA[sid] = CALL[sid](); STATUS[sid] = ("ok", "; ".join(DATA[sid].notes))
    except SourcePending as e: STATUS[sid] = ("por integrar", str(e))
    except SourceError as e: STATUS[sid] = ("sin dato", str(e)[:240])
    except Exception as e: STATUS[sid] = ("sin dato", f"{type(e).__name__}: {str(e)[:200]}")
pd.DataFrame([(k, *v) for k, v in STATUS.items()], columns=["fuente", "estado", "detalle"])

## 8. EDA en nueve etapas

Cada etapa recibe lo que cargó la sección 7, **escribe su resultado en `REPORT`** y no detiene a las demás si falla. Cada una declara su entrada, qué revisa y su salida, para que un agente pueda ejecutarla, repetirla o mejorarla por separado. Los umbrales marcados *propuesta* se calibran con un técnico local.

| Etapa | Pregunta que responde | Salida |
| --- | --- | --- |
| E0 | ¿Qué fuentes llegaron? | tabla de estados |
| E1 | ¿Cuánto cubre cada fuente en tiempo? | cobertura y huecos |
| E2 | ¿Los valores son válidos? | faltantes y valores fuera de rango |
| E3 | ¿Cómo es el año típico? | estacionalidad con p10 a p90 |
| E4 | ¿Las fuentes coinciden entre sí? | sesgo y correlación de lluvia |
| E5 | ¿A qué escala puede hablar cada dato? | nivel de soporte y píxeles limpios |
| E6 | ¿Cuánta agua entra y cuánta se pierde? | ET0, balance mensual y rachas secas |
| E7 | ¿Qué señales alimentan los cinco criterios? | meses sin cubierta, pendiente |
| E8 | ¿Qué se entrega y qué se degrada? | informe y borrador del paquete |

In [ ]:
REPORT = {"meta": {"mode": MODE, "plot": PLOT_NAME, "lon": LON, "lat": LAT, "plot_side_m": PLOT_SIDE_M, "start": START, "end": END,
                   "generated_at": NOW(), "fixture": FIXTURE}, "stages": {}, "warnings": [], "decisions": []}
if FIXTURE: REPORT["warnings"].append("DATOS DE PRUEBA: ningún valor de este informe es un resultado")

def stage(code_, title):
    def deco(fn):
        def run():
            try: out = fn(); REPORT["stages"][code_] = {"title": title, "status": "ok", **(out or {})}
            except SourcePending as e: REPORT["stages"][code_] = {"title": title, "status": "por integrar", "detalle": str(e)}
            except Exception as e: REPORT["stages"][code_] = {"title": title, "status": "error", "error": f"{type(e).__name__}: {str(e)[:200]}"}
            return REPORT["stages"][code_]
        return run
    return deco

def mark(ax):
    if FIXTURE: ax.text(.5, .5, "PRUEBA", transform=ax.transAxes, ha="center", va="center", fontsize=40, color="red", alpha=.15, rotation=25)

def runs_of_true(mask):
    "Longitudes de rachas consecutivas de True."
    out, n = [], 0
    for v in mask:
        if v: n += 1
        elif n: out.append(n); n = 0
    if n: out.append(n)
    return out

def daily_frames():
    return {k: v.df for k, v in DATA.items() if isinstance(v.df.index, pd.DatetimeIndex) and not k.startswith(("hls", "s2"))}

### E0 · Estado de las fuentes

In [ ]:
@stage("E0", "Estado de las fuentes")
def e0():
    t = pd.DataFrame([(k, *v) for k, v in STATUS.items()], columns=["fuente", "estado", "detalle"])
    return {"table": t.to_dict("records"), "ok": int((t.estado == "ok").sum()), "por_integrar": int((t.estado == "por integrar").sum()),
            "sin_dato": int((t.estado == "sin dato").sum())}
e0()

### E1 · Cobertura temporal y huecos

In [ ]:
@stage("E1", "Cobertura temporal y huecos")
def e1():
    res = {}
    exp = (pd.Timestamp(END) - pd.Timestamp(START)).days + 1
    for k, d in daily_frames().items():
        for c in d.columns:
            s = d[c]; gaps = runs_of_true(s.isna().values)
            res[f"{k}.{c}"] = {"dias_esperados": exp, "dias_con_dato": int(s.notna().sum()), "cobertura_pct": round(100 * s.notna().mean(), 2),
                               "hueco_mas_largo_dias": max(gaps) if gaps else 0, "primer_dia": str(s.first_valid_index().date()) if s.first_valid_index() is not None else None}
    for k in ("hls_stac", "s2_earthsearch"):
        if k in DATA and len(DATA[k].df):
            d = DATA[k].df; per_m = d.set_index("datetime").resample("MS").size()
            clean = d[d.cloud <= CLOUD_MAX].set_index("datetime").resample("MS").size().reindex(per_m.index, fill_value=0)
            res[k] = {"escenas": int(len(d)), "escenas_limpias": int((d.cloud <= CLOUD_MAX).sum()),
                      "meses_sin_escena_limpia": [str(i.date())[:7] for i in clean.index[clean == 0]], "cloud_max_pct": CLOUD_MAX}
    return {"coverage": res}
r1 = e1(); pd.DataFrame(r1.get("coverage", {})).T.head(20)

### E2 · Calidad: faltantes, valores de relleno y rangos físicos
Los límites son **propuesta** y amplios a propósito: buscan errores de captura, no clima raro.

In [ ]:
BOUNDS = {"PRECTOTCORR": (0, 600), "T2M": (-10, 50), "T2M_MAX": (-10, 55), "T2M_MIN": (-10, 45), "RH2M": (0, 100), "WS2M": (0, 50),
          "ALLSKY_SFC_SW_DWN": (0, 40), "precipitation_sum": (0, 600), "et0_fao_evapotranspiration": (0, 15), "precip_mm_day": (0, 600),
          "temperature_2m_max": (-10, 55), "temperature_2m_min": (-10, 45), "shortwave_radiation_sum": (0, 40)}

@stage("E2", "Calidad de los datos")
def e2():
    res = {}
    for k, d in daily_frames().items():
        res[k] = {"duplicados_indice": int(d.index.duplicated().sum())}
        for c in d.columns:
            lo, hi = BOUNDS.get(c, (-np.inf, np.inf)); s = d[c]
            res[k][c] = {"faltantes_pct": round(100 * s.isna().mean(), 2), "fuera_de_rango": int(((s < lo) | (s > hi)).sum()),
                         "min": None if s.dropna().empty else round(float(s.min()), 2), "max": None if s.dropna().empty else round(float(s.max()), 2)}
    for k in ("hls_stac", "s2_earthsearch"):
        if k in DATA and len(DATA[k].df):
            c = DATA[k].df.cloud; res[k] = {"nubes_p50": round(float(c.median()), 1), "nubes_p90": round(float(c.quantile(.9)), 1), "sin_dato_nubes": int(c.isna().sum())}
    if "soilgrids" in DATA: res["soilgrids"] = {"valores_nulos": int(DATA["soilgrids"].df[["mean"]].isna().sum().sum()), "notas": DATA["soilgrids"].notes}
    bad = [f"{k}.{c}" for k, v in res.items() for c, x in v.items() if isinstance(x, dict) and x.get("fuera_de_rango", 0) > 0]
    if bad: REPORT["warnings"].append(f"Valores fuera de rango en: {bad}")
    return {"quality": res}
e2()

### E3 · Estacionalidad: el año típico con su rango
Cada mes se resume con el percentil 10, la mediana y el percentil 90 **entre años**. Ese rango es el que luego pasa al contrato como `p10`/`p90`.

In [ ]:
def monthly_stats(s, how="sum"):
    m = s.resample("MS").agg(how)
    by = m.groupby(m.index.month)
    return pd.DataFrame({"p10": by.quantile(.1), "p50": by.median(), "p90": by.quantile(.9)})

SEAS = {}
@stage("E3", "Estacionalidad")
def e3():
    out = {}
    if "power" in DATA:
        p = DATA["power"].df
        SEAS["lluvia_power"] = monthly_stats(p["PRECTOTCORR"], "sum"); SEAS["temp_power"] = monthly_stats(p["T2M"], "mean")
        wet = SEAS["lluvia_power"]["p50"]; out["meses_mas_lluviosos"] = [int(m) for m in wet.nlargest(5).index]
        out["lluvia_mediana_mm_anual"] = round(float(wet.sum()), 0)
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    if "lluvia_power" in SEAS:
        s = SEAS["lluvia_power"]; ax[0].fill_between(s.index, s.p10, s.p90, alpha=.25, color="#0E8E65"); ax[0].plot(s.index, s.p50, color="#0B7A57")
        ax[0].set_title("Lluvia mensual (mm), p10 a p90 entre años"); ax[0].set_xlabel("mes"); mark(ax[0])
    if "temp_power" in SEAS:
        s = SEAS["temp_power"]; ax[1].fill_between(s.index, s.p10, s.p90, alpha=.25, color="#B84A00"); ax[1].plot(s.index, s.p50, color="#B84A00")
        ax[1].set_title("Temperatura media mensual (°C)"); ax[1].set_xlabel("mes"); mark(ax[1])
    fig.tight_layout(); fig.savefig(OUT / "e3_estacionalidad.png", dpi=110); plt.close(fig)
    return out
e3()

### E4 · ¿Coinciden las fuentes de lluvia?
Se compara **por mes**, no por día: las fuentes tienen resoluciones muy distintas (POWER ~55 km, IMERG ~11 km) y un día de diferencia no es un error. Se reporta también el acuerdo día lluvioso / día seco.

In [ ]:
@stage("E4", "Consistencia entre fuentes de lluvia")
def e4():
    ser = {}
    if "power" in DATA: ser["POWER"] = DATA["power"].df["PRECTOTCORR"]
    if "openmeteo" in DATA: ser["Open-Meteo"] = DATA["openmeteo"].df["precipitation_sum"]
    if "imerg" in DATA: ser["IMERG"] = DATA["imerg"].df["precip_mm_day"]
    names = list(ser); res = {}
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]; d = pd.concat([ser[a], ser[b]], axis=1, keys=[a, b]).dropna()
            if len(d) < 30: res[f"{a} vs {b}"] = {"nota": "menos de 30 días en común"}; continue
            m = d.resample("MS").sum(); wet = (d[a] >= WET_MM) == (d[b] >= WET_MM)
            res[f"{a} vs {b}"] = {"dias_comunes": int(len(d)), "corr_mensual": round(float(m[a].corr(m[b])), 3),
                                  "sesgo_pct_total": round(float(100 * (m[b].sum() - m[a].sum()) / max(m[a].sum(), 1e-9)), 1),
                                  "acuerdo_lluvioso_seco_pct": round(float(100 * wet.mean()), 1), "umbral_dia_lluvioso_mm": WET_MM}
            if abs(res[f"{a} vs {b}"]["sesgo_pct_total"]) > 25: REPORT["warnings"].append(f"Sesgo > 25 % entre {a} y {b}: revisar antes de usar ambos")
    if not res: return {"nota": "se necesitan al menos dos fuentes de lluvia con datos"}
    return {"pairs": res}
r4 = e4(); r4

### E5 · Escala: ¿a qué nivel puede afirmar algo cada dato?
Es la etapa que protege la regla de kinilum: un dato regional **no** se presenta como dato de la parcela.

In [ ]:
@stage("E5", "Nivel de soporte y píxeles limpios")
def e5():
    t = CATALOG[["id", "nombre", "res_nativa_m", "soporte"]].copy()
    t["estado"] = t["id"].map(lambda i: STATUS.get(i, ("no habilitada", ""))[0])
    t["pixeles_limpios_en_parcela"] = t["res_nativa_m"].map(lambda r: clean_pixels(PLOT_SIDE_M, r))
    def veredicto(row):
        if row.estado != "ok": return "sin dato"
        if row.soporte == "parcela" and row.pixeles_limpios_en_parcela == 0: return "baja a entorno"
        return row.soporte
    t["soporte_efectivo"] = t.apply(veredicto, axis=1)
    return {"table": t.to_dict("records"), "regla": "tras descartar un borde de 30 m, si no queda ningún píxel completo, el dato baja de parcela a entorno"}
r5 = e5(); pd.DataFrame(r5.get("table", []))

### E6 · Balance de agua preliminar
ET0 por **FAO-56 Penman–Monteith** con las variables de POWER, comparada con la ET0 de Open-Meteo cuando está. Supuestos que se declaran: `G = 0`, albedo 0.23, humedad del aire calculada con la humedad relativa media y la temperatura media, elevación `ELEV_M`. El resultado es **preliminar**: sirve para ver si el orden de magnitud es razonable, no para decidir.

In [ ]:
def es_kpa(t): return 0.6108 * np.exp(17.27 * t / (t + 237.3))

def et0_pm(df, lat, elev, rs_units="MJ/m^2/day"):
    "FAO-56 Penman-Monteith diaria (mm/día). Columnas de POWER: T2M, T2M_MAX, T2M_MIN, RH2M, WS2M, ALLSKY_SFC_SW_DWN."
    t, tx, tn = df["T2M"], df["T2M_MAX"], df["T2M_MIN"]
    rs = df["ALLSKY_SFC_SW_DWN"] * (3.6 if "kW" in str(rs_units) else 1.0)
    u2 = df["WS2M"]; ea = df["RH2M"] / 100 * es_kpa(t); es = (es_kpa(tx) + es_kpa(tn)) / 2
    delta = 4098 * es_kpa(t) / (t + 237.3) ** 2
    P = 101.3 * ((293 - 0.0065 * elev) / 293) ** 5.26; gamma = 0.000665 * P
    J = df.index.dayofyear.values; phi = math.radians(lat)
    dr = 1 + 0.033 * np.cos(2 * np.pi * J / 365); dec = 0.409 * np.sin(2 * np.pi * J / 365 - 1.39)
    ws = np.arccos(np.clip(-np.tan(phi) * np.tan(dec), -1, 1))
    ra = (24 * 60 / np.pi) * 0.0820 * dr * (ws * np.sin(phi) * np.sin(dec) + np.cos(phi) * np.cos(dec) * np.sin(ws))
    rso = (0.75 + 2e-5 * elev) * ra
    ratio = np.clip(rs.values / np.where(rso > 0, rso, np.nan), 0.3, 1.0)
    rns = 0.77 * rs.values
    rnl = 4.903e-9 * (((tx + 273.16) ** 4 + (tn + 273.16) ** 4) / 2).values * (0.34 - 0.14 * np.sqrt(ea.values)) * (1.35 * ratio - 0.35)
    rn = rns - rnl
    et0 = (0.408 * delta.values * rn + gamma * 900 / (t.values + 273) * u2.values * (es - ea).values) / (delta.values + gamma * (1 + 0.34 * u2.values))
    return pd.Series(np.clip(et0, 0, None), index=df.index, name="ET0_PM_mm")

ET0 = {}
@stage("E6", "Balance de agua preliminar")
def e6():
    if "power" not in DATA: raise SourcePending("POWER no llegó: sin ET0 propia")  # la etapa queda como "por integrar", no como error
    p = DATA["power"]; units = p.units.get("ALLSKY_SFC_SW_DWN", "MJ/m^2/day")
    et = et0_pm(p.df, LAT, ELEV_M, units); ET0["pm"] = et
    rain = p.df["PRECTOTCORR"]; bal = (rain - et).resample("MS").sum(); out = {"unidades_radiacion": units}
    out["et0_mediana_mm_dia"] = round(float(et.median()), 2); out["et0_anual_mediana_mm"] = round(float(et.resample("YS").sum().median()), 0)
    out["balance_mensual_p50_mm"] = {int(m): round(float(v), 1) for m, v in bal.groupby(bal.index.month).median().items()}
    out["meses_con_balance_negativo_p50"] = [int(m) for m, v in bal.groupby(bal.index.month).median().items() if v < 0]
    spells = []
    for y, s in (rain < WET_MM).groupby(rain.index.year): r = runs_of_true(s.values); spells.append(max(r) if r else 0)
    out["racha_seca_mas_larga_por_anio_dias"] = {"p10": float(np.percentile(spells, 10)), "p50": float(np.median(spells)), "p90": float(np.percentile(spells, 90))}
    out["supuestos"] = ["G = 0", "albedo 0.23", f"elevación {ELEV_M} m (supuesto hasta integrar NASADEM)", "ea con RH media y T media"]
    if "openmeteo" in DATA:
        c = pd.concat([et, DATA["openmeteo"].df["et0_fao_evapotranspiration"]], axis=1, keys=["pm", "om"]).dropna()
        if len(c) > 30:
            out["vs_openmeteo"] = {"corr_diaria": round(float(c.pm.corr(c.om)), 3), "sesgo_pct": round(float(100 * (c.pm.sum() - c.om.sum()) / c.om.sum()), 1)}
            if abs(out["vs_openmeteo"]["sesgo_pct"]) > 20: REPORT["warnings"].append("ET0 propia difiere >20 % de la de Open-Meteo: revisar unidades y elevación")
    fig, ax = plt.subplots(figsize=(6.5, 3.2)); ax.bar(range(1, 13), [out["balance_mensual_p50_mm"].get(m, 0) for m in range(1, 13)], color="#0E8E65")
    ax.axhline(0, color="#0A0A0A", lw=.8); ax.set_title("Lluvia menos ET0, mediana por mes (mm)"); ax.set_xlabel("mes"); mark(ax)
    fig.tight_layout(); fig.savefig(OUT / "e6_balance.png", dpi=110); plt.close(fig)
    return out
e6()

### E7 · Señales para los cinco criterios
Por ahora solo hay una señal con datos: los **meses sin cubierta verde** a partir de NDVI. El umbral es **propuesta** y se calibra con un técnico. La pendiente depende de NASADEM, que sigue como esqueleto.

In [ ]:
NDVI_COVER_MIN = 0.35   # propuesta, por calibrar
@stage("E7", "Señales para los criterios")
def e7():
    out = {"agua": "ver E6", "riesgo_climatico": "ver E6 (rachas secas)"}
    if "modis" in DATA:
        n = DATA["modis"].df["ndvi"].resample("MS").mean(); low = n < NDVI_COVER_MIN
        by = low.groupby(low.index.month).mean()
        out["cubierta"] = {"umbral_ndvi": NDVI_COVER_MIN, "fraccion_de_anios_sin_cubierta_por_mes": {int(m): round(float(v), 2) for m, v in by.items()},
                           "meses_sin_cubierta_tipicos": [int(m) for m, v in by.items() if v >= .5]}
    else: out["cubierta"] = "por integrar: falta serie de NDVI (MODIS o HLS con píxeles)"
    out["erosion"] = "por integrar: falta pendiente de NASADEM" if "nasadem" not in DATA else "pendiente"
    out["suelo"] = "SoilGrids disponible" if "soilgrids" in DATA else "por integrar"
    out["ingreso"] = "fuera de este notebook: precios y costos vienen de fuentes locales (SIAP, INIFAP)"
    return out
e7()

### E8 · Informe, paquete de datos y plan de degradación

El plan de degradación aplica la tabla de reglas del flujo de punta a punta: qué se hace cuando una fuente no llega. **El paquete es un borrador**: lleva solo valores con su contrato; en modo `fixture` todo queda marcado `PRUEBA`.

In [ ]:
CORE = ["power", "imerg", "hls_stac", "nasadem"]
def degradation_plan(status):
    ok = lambda s: status.get(s, ("sin dato",))[0] == "ok"
    plan, stop = [], False
    if not ok("grace"): plan.append(("grace", "omitir; nada visible cambia", "ninguno"))
    if not ok("smap"): plan.append(("smap", "humedad de contexto desde balance hídrico con POWER y GPM", "Humedad estimada"))
    if not ok("ecostress"): plan.append(("ecostress", "calor desde POWER", "Evidencia de parcela no disponible"))
    if not ok("modis"): plan.append(("modis", "usar VIIRS si está en el catálogo; si no, marcar por integrar", "según el caso"))
    if not ok("hls_stac"):
        if ok("modis") or ok("s2_earthsearch"): plan.append(("hls_stac", "bajar soporte a entorno con MODIS o Sentinel-2", "Entorno"))
        else: plan.append(("hls_stac", "SIN RESPALDO: detener entrega", "—")); stop = True
    if not ok("power"): plan.append(("power", "SIN RESPALDO: detener entrega", "—")); stop = True
    if not ok("nasadem"): plan.append(("nasadem", "SIN RESPALDO: detener entrega", "—")); stop = True
    if not ok("imerg"):
        if ok("openmeteo"): plan.append(("imerg", "hay contraste (Open-Meteo); decidir con el equipo si lo sustituye. Mientras tanto no se publica como fuente de zona", "Lluvia de zona no disponible"))
        else: plan.append(("imerg", "SIN RESPALDO: detener entrega", "—")); stop = True
    return plan, stop

@stage("E8", "Informe, paquete y degradación")
def e8():
    plan, stop = degradation_plan(STATUS)
    REPORT["decisions"] = [{"fuente": f, "accion": a, "aviso_en_pantalla": av} for f, a, av in plan]
    REPORT["decision_entrega"] = "DETENER: falta una fuente del núcleo sin respaldo" if stop else "PUBLICAR con avisos"
    recs = []
    if "power" in DATA and "lluvia_power" in SEAS:
        pr = DATA["power"].prov
        for m, r in SEAS["lluvia_power"].iterrows():
            recs.append(make_record(f"lluvia_mensual_mes{m:02d}", r.p50, "mm", r.p10, r.p90, pr, 55000, "modelado",
                                    assumptions=["p10 y p90 entre años del mismo mes", "resolución regional, no de parcela"]))
    if "soilgrids" in DATA:
        s = DATA["soilgrids"]; row = s.df[(s.df.property == "clay") & (s.df.depth == "0-5cm")]
        if len(row):
            r = row.iloc[0]; recs.append(make_record("arcilla_0_5cm", r["mean"], "%", r["Q0.05"], r["Q0.95"], s.prov, 250, "modelado",
                                                     assumptions=["rango = cuantiles 5 y 95", "mapa global, no medición de la parcela"]))
    pkg = {"fixture": FIXTURE, "plot": PLOT_NAME, "records": [asdict(x) for x in recs]}
    (OUT / "data_package_draft.json").write_text(json.dumps(pkg, ensure_ascii=False, indent=1, default=str))
    kb = round(len(json.dumps(pkg, default=str)) / 1024, 1)
    return {"registros_en_paquete": len(recs), "tamano_kb_borrador": kb, "tope_kb_del_paquete_final": 500}
r8 = e8()

(OUT / "eda_report.json").write_text(json.dumps(REPORT, ensure_ascii=False, indent=1, default=str))
lines = [f"# Informe EDA · {PLOT_NAME}", "", f"Modo: **{MODE}** · {REPORT['meta']['generated_at']}", ""]
if FIXTURE: lines += ["> **DATOS DE PRUEBA. Nada de este informe es un resultado.**", ""]
lines += [f"Decisión de entrega: **{REPORT.get('decision_entrega')}**", "", "## Fuentes", ""]
lines += [f"- {k}: {v[0]}" + (f" ({v[1]})" if v[1] else "") for k, v in STATUS.items()]
lines += ["", "## Avisos", ""] + ([f"- {w}" for w in REPORT["warnings"]] or ["- ninguno"])
lines += ["", "## Etapas", ""] + [f"- {k} · {v['title']}: {v['status']}" for k, v in REPORT["stages"].items()]
(OUT / "eda_report.md").write_text("\n".join(lines))
print("\n".join(lines)); print("\nArchivos en", OUT, ":", sorted(p.name for p in OUT.iterdir()))

## 9. Qué sigue (para quien tome este notebook)

1. Ejecutar en modo `live` desde una máquina con salida a internet y guardar `out/eda_report.json` en el repositorio.
2. Integrar los conectores con credencial en este orden: **IMERG, NASADEM, MODIS** (núcleo y quinta fuente), después SMAP, ECOSTRESS y GRACE.
3. Mover las secciones 2 a 6 a un paquete `kinilum_data/` (un módulo por fuente) y dejar este notebook como el orquestador del EDA.
4. Calibrar con un técnico local: `CLOUD_MAX`, `WET_MM`, `NDVI_COVER_MIN` y los límites de la sección E2.
5. Revisar con el equipo las **licencias y atribuciones** de cada fuente antes de mostrarlas en la herramienta.